In [ ]:
from openai import OpenAI

openai_client = OpenAI()
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="are there companies in nyc working with AI that are actually PRO HUMAN WORKER? What are they? ARe they hiring?"
)

In [15]:
response.output_text

'Yes, there are several companies in New York City that focus on using AI in ways that are designed to complement human workers rather than replace them. Here are a few examples of such companies:\n\n1. **IBM**: IBM has been actively integrating AI into its products to enhance human decision-making, particularly in healthcare and business analytics. They often emphasize the importance of human oversight in AI.\n\n2. **UiPath**: This company specializes in robotic process automation (RPA) that works alongside human workers to automate repetitive tasks, allowing humans to focus on more complex and creative work.\n\n3. **Zocdoc**: While primarily a healthcare appointment scheduling platform, Zocdoc uses AI to improve patient experiences by assisting human staff in customer service and administrative tasks.\n\n4. **Gusto**: This payroll and HR platform integrates AI to simplify human resource tasks, allowing employees to focus more on people-centric work rather than administrative duties.\

In [17]:
response.usage

ResponseUsage(input_tokens=33, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=371, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=404)

In [ ]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5-nano": {"input": 0.075, "output": 0.30},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5.2": {"input": 1.75, "output": 14.00},
    "gpt-5.2-pro": {"input": 21.00, "output": 168.00},
    "claude-haiku-4.5": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4.6": {"input": 3.00, "output": 15.00},
    "claude-opus-4.7": {"input": 5.00, "output": 25.00},
    "claude-opus-4.6": {"input": 5.00, "output": 25.00},
    "claude-opus-4.5": {"input": 5.00, "output": 25.00},
    "claude-sonnet-4.5": {"input": 3.00, "output": 15.00},
    "claude-3.5-haiku": {"input": 0.80, "output": 4.00},
    "claude-3-haiku": {"input": 0.25, "output": 1.25},
}
# Extend with anthropic costs and wire up in CalculateCosts


def calculate_cost(model_name: str, input_tokens: int, output_tokens: int) -> float:
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost

usage = response.usage

cost = calculate_cost(
    model_name="gpt-4o-mini",
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens
)

print(f"Cost: ${cost:.6f}")

Cost: $0.000228


In [13]:
response.model_dump_json(indent=2)

'{\n  "id": "resp_0559ca8310b2e5930069e7f3cd08e8819d9227a1e60ea5e0fb",\n  "created_at": 1776808909.0,\n  "error": null,\n  "incomplete_details": null,\n  "instructions": null,\n  "metadata": {},\n  "model": "gpt-4o-mini-2024-07-18",\n  "object": "response",\n  "output": [\n    {\n      "id": "msg_0559ca8310b2e5930069e7f3ce7f4c819dae05261e009b9f7a",\n      "content": [\n        {\n          "annotations": [],\n          "text": "Yes, there are several companies in New York City that focus on using AI in ways that are designed to complement human workers rather than replace them. Here are a few examples of such companies:\\n\\n1. **IBM**: IBM has been actively integrating AI into its products to enhance human decision-making, particularly in healthcare and business analytics. They often emphasize the importance of human oversight in AI.\\n\\n2. **UiPath**: This company specializes in robotic process automation (RPA) that works alongside human workers to automate repetitive tasks, allowin

In [7]:
import os
from typing import Literal
from dotenv import load_dotenv
from pathlib import Path
from anthropic import Anthropic
from openai import OpenAI

Provider = Literal["anthropic", "openai"]

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

load_dotenv(REPO_ROOT / ".env")

def get_client(provider: Provider):
    if provider == "anthropic":
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("Missing ANTHROPIC_API_KEY")
        return Anthropic(api_key=api_key)

    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("Missing OPENAI_API_KEY")
        return OpenAI(api_key=api_key)

    raise ValueError(f"Unsupported provider: {provider}")


def get_default_model(provider: Provider) -> str:
    if provider == "anthropic":
        return "claude-sonnet-4-6"
    if provider == "openai":
        return "gpt-5.4"
    raise ValueError(f"Unsupported provider: {provider}")

def get_cheapest_model(provider: Provider) -> str:
    if provider == "anthropic":
        return "claude-haiku-4-5"
    if provider == "openai":
        return "gpt-5.4-mini"
    raise ValueError(f"Unsupported provider: {provider}")    


def extract_anthropic_text(response) -> str:
    parts = []
    for block in response.content:
        text = getattr(block, "text", None)
        if text:
            parts.append(text)
    return "\n".join(parts).strip()


def extract_openai_text(response) -> str:
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text.strip()

    parts = []
    for item in getattr(response, "output", []):
        for content in getattr(item, "content", []):
            text = getattr(content, "text", None)
            if text:
                parts.append(text)
    return "\n".join(parts).strip()


def ask_model(
    prompt: str,
    provider: Provider,
    model: str | None = None,
    max_tokens: int = 400,
) -> str:
    client = get_client(provider)
    #model = model or get_default_model(provider)
    model = model or get_cheapest_model(provider)

    if provider == "anthropic":
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return extract_anthropic_text(response)

    if provider == "openai":
        response = client.responses.create(
            model=model,
            input=prompt,
            max_output_tokens=max_tokens,
        )
        return extract_openai_text(response)

    raise ValueError(f"Unsupported provider: {provider}")

In [ ]:
print(ask_model("Say hello from Claude.", provider="anthropic", model="claude-haiku-4-5"))
#print(ask_model("Say hello from Claude.", provider="openai", model="gpt-5.4-mini"))

Hello! I'm Claude, an AI assistant made by Anthropic. It's nice to meet you! How can I help you today?


In [ ]:
ant_response = ask_model("isn't agi basically impossible?  convince me", provider="anthropic")

In [ ]:
print(ant_response)

'# Why AGI might be genuinely hard\n\n**The honest case:**\n\n- **We don\'t know what we\'re missing.** We have no agreed theory of general intelligence. Current ML is pattern-matching at scale; we can\'t point to what makes that "general." It\'s like trying to build a plane while still debating what flight actually is.\n\n- **Scaling hits walls.** Every major breakthrough (RNNs → Transformers → bigger transformers) looked like a solution until it wasn\'t. Each required architectural rethinking. We might be in that cycle indefinitely.\n\n- **Fundamental gaps are real:**\n  - Current systems can\'t plan more than a few steps ahead reliably\n  - No genuine abstraction/symbol manipulation (they pattern-match symbols)\n  - No causal reasoning\n  - No understanding of physics, agency, or consequence\n  - Sample efficiency is abysmal compared to humans\n\n- **The bar is impossibly high.** AGI means: reasoning across domains, learning from few examples, genuine transfer learning, knowing what

In [ ]:
ant_response.usage

ResponseUsage(input_tokens=33, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=371, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=404)